# WS6 — Deep sequence GRU on a free Colab **T4** (Phase-2 GPU path)

This is the **self-contained Colab path** for Workstream 6 (decision D46). It trains the
**O-view GRU** (the ordered-sequence encoder) on a T4 GPU with the *same* `pitchseq` / WS6
code Sean runs locally, and prints the headline numbers to paste back.

It is **not** the WS6 research notebook (that is `notebooks/ws6_deep_seq.ipynb`, built in WS6b).
This one does one job: run the compute-peak recurrent fit fast on a GPU.

**When to use this instead of Step WS6.1 (local CPU):** the recurrent O/OM fits are the one
place a GPU helps (~10–20×). On a T4 the full-data O-view fit is roughly **~1 hour** vs several
hours on a desktop CPU. Everything else in the study is CPU work and does not need this.

## ⚠️ AMD GPU caveat (ROCm / DirectML) — why Colab, not the desktop GPU

Sean's desktop GPU is **AMD**. PyTorch's Windows wheels are **CPU/CUDA only** — there is no
stable Windows **ROCm** build, and `torch-directml` is an unofficial, frequently-lagging backend
that does not reliably support the ops used here. **Do not fight the AMD GPU.** Use either:

- **Step WS6.1 — local CPU** (`pip install -e ".[deep]"`, `--device auto` → CPU): honest, simple,
  slow; or
- **this notebook — free Colab T4**: the recommended fast route.

WS6 is the only GPU-relevant step in the study, and only its *optional* Transformer demo is
genuinely GPU-preferred.

## 0. Select the GPU runtime

In the Colab menu: **Runtime → Change runtime type → Hardware accelerator = T4 GPU**, then run
the cell below. It should print a CUDA device; if it prints `cpu`, the runtime is not on a GPU.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — set Runtime → Change runtime type → T4 GPU, then re-run.')

## 1. Bring in the repo + install the `[deep]` extra

Colab ships a recent PyTorch, so we only need the repo and its deps. **Fill in Sean's repo URL**
below (the private/public GitHub URL of `pitch-sequencing-research`). If the repo is private,
use a token URL or upload a zip instead (see the commented alternative).

In [ ]:
# --- Option A: clone the repo (fill in the URL) -------------------------------------
REPO_URL = 'https://github.com/<SEAN_USER>/pitch-sequencing-research.git'  # <-- EDIT ME
import os, subprocess, sys
if not os.path.isdir('pitch-sequencing-research'):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
%cd pitch-sequencing-research
# Install the package with the deep extra (torch is already present on Colab).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[deep]', '-q'], check=True)

# --- Option B (private repo / no git): upload a repo zip via the Files panel and unzip -
# !unzip -q pitch-sequencing-research.zip && %cd pitch-sequencing-research && pip install -e '.[deep]' -q

## 2. Get the decision table onto the runtime

The full `decision_table.parquet` is built locally in **Step 1** of the runbook and lives on
Sean's desktop. Two ways to get it here:

- **Mount Google Drive** and point at a copy you uploaded to Drive (recommended for the full
  ~3.85M-row table), or
- **skip the real table** and run a small **synthetic** world end-to-end first to prove the GPU
  path works (fast; a good smoke test before committing the full upload).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# EDIT to the Drive path where you uploaded decision_table.parquet:
TABLE_PATH = '/content/drive/MyDrive/pitchseq/decision_table.parquet'
DRIVE_OUT = '/content/drive/MyDrive/pitchseq/ws6_results'  # artifacts + runmeta land here
import os; os.makedirs(DRIVE_OUT, exist_ok=True)

### 2b. (optional) smoke-test the GPU path on a synthetic world first

Run this to confirm the whole path works on the GPU before uploading the real table. It builds a
small null world itself and trains the O-view GRU on it (~a minute on a T4).

In [ ]:
from workstreams.ws6_deep_seq.run_ws6 import run_ws6, _format_headline
smoke = run_ws6(synth='null', out='/content/ws6_smoke', views=['C','U','L1','O'],
                targets=['selection','outcome1'], n_games=150, epochs=25, device='auto',
                n_perm=6, write_outputs=True)
print(_format_headline(smoke))

## 3. Train the O-view GRU on the **full** data (GPU)

Same `run_ws6` pipeline as local, `--views O` (add `U L1 C` if you also want the full ablation
here), `device='auto'` → the T4. Checkpoints per `(view, target)` land in `DRIVE_OUT`, so a
disconnected runtime resumes on re-run. Expect **~1 hour** for O on the full table.

In [ ]:
result = run_ws6(
    source=TABLE_PATH,          # the full decision table on Drive
    synth='off',
    out=DRIVE_OUT,              # artifacts + runmeta saved back to Drive (resumable)
    views=['C', 'U', 'L1', 'O'],  # the ablation ladder; add 'OM' for matchup memory
    targets=['selection', 'outcome1'],
    epochs=30,
    hidden=64, embed=16, max_len=15, batch=512,
    device='auto',             # -> cuda on the T4
    n_boot=200, n_perm=0,      # permutation refits are a synthetic-only gate; skip on real data
    write_outputs=True,
)

## 4. Print the headline to paste back

Copy the whole block below into the review thread, plus the Drive path of the saved artifacts.

In [ ]:
print(_format_headline(result))
print('\nArtifacts + runmeta saved to:', DRIVE_OUT)
import glob
print('files:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(DRIVE_OUT, '*'))))

## 5. What to paste back

1. the printed **headline block** (central table, both Delta_order lines with their CIs, the
   **locked-test** outcome1 Delta_order, and the **Pareto row** — params / epochs / wall-clock),
2. the **Drive path** of the saved artifacts (so the numbers can be filled into the papers), and
3. the wall-clock from the runmeta (`ws6_real.runmeta.json` in `DRIVE_OUT`).

Read the result with the **D21 rule** (`Delta_order` is negatively biased under the null → the
criterion is "not significantly positive"; a small negative is *consistent with no ordering
effect*, never "order hurts"), and against **WS2/WS3**: does the *learned* O representation beat
the engineered tabular history (WS3 GBDT-O) and the explicit grammar (WS2)? Either answer is a
clean finding (SPEC §13).